In [2]:
clear_train_dir = False

DATASET = "6DMG"
num_label = 20
EPOCHS = 300
BATCH_SIZE = 64

# list of layers: [numcores; axon; neuron]
model_configs = [[13, 238, 64], [4, 208 ,64], [1, 256, 240]]
# list of config for each layer: [thres, activation_factor]
network_activation = [[0,0.7], [0,0.6], [0,0.8]]

# Path

In [3]:
# Set up base dirs
import os

ROOT_DIR = os.getcwd()

SOFT_DIR=ROOT_DIR+"/Software"
HARD_DIR=ROOT_DIR+"/Hardware"

DATASET_DIR = SOFT_DIR+"/data/processed/"
TRAIN_DIR=SOFT_DIR+"/training"
LOG_DIR=SOFT_DIR+"/log/"+DATASET

os.makedirs(LOG_DIR, exist_ok=True)

# Install

In [4]:
!pip install tensorflow
!pip install keras

In [5]:
%cd {SOFT_DIR}

if (clear_train_dir):
    !rm -rf {TRAIN_DIR}
    !unzip "training.zip"

/home/nam/NPLink/SNN_framework/Software


/home/nam/anaconda3/envs/duongk65/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
%cd {TRAIN_DIR}
!pip install "./tealayers/tealayer2.0"
!pip install "./edabkutils"

/home/nam/NPLink/SNN_framework/Software/training


Processing ./tealayers/tealayer2.0
  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'tealayer2' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'tealayer2'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for tealayer2: filename=tealayer2-2.0-py3-none-any.whl size=8870 sha256=1983abc75fdeae0207079cdf6c24ac62a9c79c21e9c61ee03230ec7bbe7d8eda
  Stored in directory: /tmp/pip-ephem-wheel-cache-5mty6h6l/wheels/ce/93/6e/c37f23f189b3ea91f3b86410fa832725009716c2170b2f859c
Successfully built tealayer2
  Attempting uninstall: tealayer2
    Found existing installation: tealayer2 2.0
    Uninstalling tealayer2-2.0:
      Successfully uninstalled tealayer2-

# Train

## Prepare and import package

In [7]:
from tealayer2 import Tea, AdditivePooling, tea_weight_initializer
from tensorflow.keras.layers import Flatten, Activation, Input, Lambda, Concatenate
from tensorflow.keras.losses import CategoricalFocalCrossentropy,BinaryCrossentropy, CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras import Model
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, ModelCheckpoint

import json
import yaml
import numpy as np
import math
import tensorflow.compat.v1 as tf
from tensorflow.keras.optimizers import Adam
from sklearn.utils import class_weight

from edabkutils.modelize import auto_train_config, save_configure_json, get_configs, get_core_arrange, write_config_sim

2026-03-10 16:43:14.614926: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-10 16:43:15.631416: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [8]:
number_of_layers = len(model_configs)
[x_range, y_range, core_arrange] = get_core_arrange(model_configs)

print("Model configurations:", model_configs)
print("Core arrangement:", core_arrange)

Model configurations: [[13, 238, 64], [4, 208, 64], [1, 256, 240]]
Core arrangement: [[[0, 0], [1, 0], [2, 0], [3, 0], [4, 0], [5, 0], [6, 0], [7, 0], [8, 0], [9, 0], [10, 0], [11, 0], [12, 0]], [[0, 1], [1, 1], [2, 1], [3, 1]], [[0, 2]]]


In [9]:
data = np.load(DATASET_DIR+DATASET+".npz")

X_train = data["X_train"]
y_train = data["y_train"]

X_test = data["X_test"]
y_test = data["y_test"]

class_weights = class_weight.compute_class_weight(class_weight='balanced',
                                                 classes=np.unique(y_train),
                                                 y=y_train)
print(f"Class weight: {class_weights}")

y_train = to_categorical(y_train, num_label)
y_test = to_categorical(y_test, num_label)

X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

Class weight: [0.99822222 0.98508772 1.03502304 0.93974895 0.96810345 1.03027523
 0.98942731 1.03502304 1.02090909 0.93974895 1.00717489 0.97229437
 1.03502304 1.01628959 0.96810345 0.99380531 1.02557078 0.97652174
 1.05446009 1.03027523]


## Train model

In [ ]:
# # Initial the SNN network

# # Shape the input to right size
# inputs = Input(shape=(X_train.shape[1:]))
# # print(f"inputs = Input(shape={(X_train.shape[1:])})")
# # print(f"core_size = {model_configs[0][1]}")

# # Flatten the inputs
# flattened_inputs = Flatten()(inputs)

# layer_input = flattened_inputs
# # For loop for each layer
# for layer_ind in range(number_of_layers-1):
#   layer = []
#   # For each core in each layer
#   for core_ind in range(model_configs[layer_ind][0]):
#     core = Lambda(lambda x, start=model_configs[layer_ind][1]*core_ind, end=model_configs[layer_ind][1]*(core_ind+1): x[:, start:end])(layer_input)
#     core = Tea(units=model_configs[layer_ind][2], threshold = network_activation[layer_ind][0], activation_factor = network_activation[layer_ind][1], name=f'tea_{layer_ind}_{core_ind}')(core)
#     # print(f"core = Tea(units={model_configs[layer_ind][2]}, threshold = {network_activation[layer_ind][0]}, activation_factor = {network_activation[layer_ind][1]}, name=f'tea_{layer_ind}_{core_ind}')(core)")
#     layer.append(core)

#   layer_input = Concatenate(axis=1)(layer)
#   # print(f"layer_input = Concatenate(axis=1)({layer})")

# core = Tea(units=model_configs[number_of_layers-1][2], threshold=network_activation[number_of_layers-1][0], activation_factor=network_activation[number_of_layers-1][1], name=f'tea_{number_of_layers-1}')(layer_input)
# # print(f"core = Tea(units={model_configs[number_of_layers-1][2]}, threshold={network_activation[number_of_layers-1][0]}, activation_factor={network_activation[number_of_layers-1][1]}, name=f'tea_{number_of_layers-1}')(layer_input)")
# network = AdditivePooling(num_label)(core)

In [1]:
def build_model(model_configs, network_activation, num_label, input_shape):

    number_of_layers = len(model_configs)

    inputs = Input(shape=input_shape)
    flattened_inputs = Flatten()(inputs)

    layer_input = flattened_inputs

    for layer_ind in range(number_of_layers-1):

        layer = []

        for core_ind in range(model_configs[layer_ind][0]):

            core = Lambda(
                lambda x,
                start=model_configs[layer_ind][1]*core_ind,
                end=model_configs[layer_ind][1]*(core_ind+1):
                x[:, start:end]
            )(layer_input)

            core = Tea(
                units=model_configs[layer_ind][2],
                threshold=network_activation[layer_ind][0],
                activation_factor=network_activation[layer_ind][1],
                name=f'tea_{layer_ind}_{core_ind}'
            )(core)

            layer.append(core)

        layer_input = Concatenate(axis=1)(layer)

    core = Tea(
        units=model_configs[-1][2],
        threshold=network_activation[-1][0],
        activation_factor=network_activation[-1][1],
        name=f'tea_{number_of_layers-1}'
    )(layer_input)

    network = AdditivePooling(num_label)(core)

    predictions = Activation('softmax')(network)

    model = Model(inputs=inputs, outputs=predictions)

    model.compile(
        loss=CategoricalCrossentropy(),
        optimizer=Adam(),
        metrics=['accuracy'],
        run_eagerly=True
    )

    return model

## Optuna

In [ ]:
import optuna
import numpy as np
from optuna.integration import TFKerasPruningCallback

input_dim = np.prod(X_train.shape[1:])
num_layers = 2


def objective(trial):

    model_configs = []
    network_activation = []

    # ======================
    # TRAINING PARAMETERS
    # ======================

    lr = trial.suggest_float(
        "learning_rate",
        1e-5,
        1e-2,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [16, 32, 64, 128]
    )

    # ======================
    # LAYER 0 (INPUT)
    # ======================

    core0 = trial.suggest_int("core0", 1, 32)

    # enforce input constraint
    if input_dim % core0 != 0:
        raise optuna.TrialPruned()

    axon0 = input_dim // core0

    neuron0 = trial.suggest_int(
        "neuron0",
        32,
        256
    )

    th0 = trial.suggest_int("th0", 0, 5)

    af0 = trial.suggest_float(
        "af0",
        0.1,
        1
    )

    model_configs.append([core0, axon0, neuron0])
    network_activation.append([th0, af0])

    # connection size
    prev_connection = core0 * neuron0

    # ======================
    # HIDDEN LAYERS
    # ======================

    for i in range(1, num_layers):

        core_i = trial.suggest_int(f"core{i}", 1, 32)

        neuron_i = trial.suggest_int(
            f"neuron{i}",
            32,
            256
        )

        th = trial.suggest_int(f"th{i}", 0, 5)

        af = trial.suggest_float(
            f"af{i}",
            0.1,
            1
        )

        connection = core_i * neuron_i

        # enforce topology constraint
        if prev_connection % core_i != 0:
            raise optuna.TrialPruned()

        axon_i = prev_connection // core_i

        model_configs.append([core_i, axon_i, neuron_i])
        network_activation.append([th, af])

        prev_connection = connection

    # ======================
    # OUTPUT CONSTRAINT
    # ======================

    # enforce neuron_last = k * num_label
    k = trial.suggest_int("output_multiplier", 1, 20)
    model_configs[-1][2] = k * num_label

    # ======================
    # BUILD MODEL
    # ======================

    model = build_model(
        model_configs,
        network_activation,
        num_label,
        X_train.shape[1:]
    )

    model.compile(
        loss=CategoricalCrossentropy(),
        optimizer=Adam(learning_rate=lr),
        metrics=["accuracy"],
        run_eagerly=True
    )


    # ======================
    # TRAIN
    # ======================

    # Using callback EarlyStopping
    early_stopping = EarlyStopping(
        monitor='val_accuracy',  # monitor the accuracy of validation set
        patience=5,  # Allow max 5 epoch without improvement
        min_delta=0,
        mode='max',
        verbose=1,
        restore_best_weights=True,
    )

    prun_call = TFKerasPruningCallback(trial, "val_accuracy")


    history = model.fit(
        X_train,
        y_train,
        batch_size=batch_size,
        epochs=100,
        validation_split=0.2,
        verbose=0,
        callbacks=[prun_call, early_stopping]
    )

    val_acc = max(history.history["val_accuracy"])

    trial.set_user_attr(
        "model_configs",
        [[int(a), int(b), int(c)] for a,b,c in model_configs]
    )

    trial.set_user_attr(
        "network_activation",
        [[int(th), float(af)] for th,af in network_activation]
    )

    return val_acc

In [40]:
study = optuna.create_study(
    study_name="tealayer_snn_search",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.MedianPruner(),
    storage="sqlite:///optuna_snn.db",
    load_if_exists=True
)

import json
import os

def log_trial_callback(study, trial):

    if trial.state != optuna.trial.TrialState.COMPLETE:
        return

    result = {
        "trial_id": trial.number,
        "value": trial.value,
        "params": trial.params,
        "user_attrs": trial.user_attrs
    }

    file = "trial_log.json"

    if os.path.exists(file):

        with open(file, "r") as f:
            data = json.load(f)

    else:
        data = []

    data.append(result)

    with open(file, "w") as f:
        json.dump(data, f, indent=4)

[I 2026-03-10 17:29:43,685] Using an existing study with name 'tealayer_snn_search' instead of creating a new one.


In [41]:
study.optimize(objective, n_trials=200, callbacks=[log_trial_callback])

[I 2026-03-10 17:29:46,857] Trial 14 pruned. 


Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 1.


[W 2026-03-10 17:31:06,865] Trial 15 failed with parameters: {'learning_rate': 0.0001075425510496464, 'batch_size': 16, 'core0': 1, 'neuron0': 176, 'th0': 5, 'af0': 0.9772375508904649, 'core1': 1, 'neuron1': 38, 'th1': 3, 'af1': 0.8916203784984277, 'output_multiplier': 2} because of the following error: TypeError('Object of type int64 is not JSON serializable').
Traceback (most recent call last):
  File "/home/nam/anaconda3/envs/duongk65/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_525577/1862823798.py", line 153, in objective
    trial.set_user_attr("model_configs", model_configs)
  File "/home/nam/anaconda3/envs/duongk65/lib/python3.10/site-packages/optuna/trial/_trial.py", line 597, in set_user_attr
    self.storage.set_trial_user_attr(self._trial_id, key, value)
  File "/home/nam/anaconda3/envs/duongk65/lib/python3.10/site-packages/optuna/storages/_cached_storage.py", line 202, in set_trial

TypeError: Object of type int64 is not JSON serializable

## Train

In [10]:
existing_runs = [
    d for d in os.listdir(LOG_DIR)
    if d.startswith("run_")
]

run_numbers = [int(d.split("_")[1]) for d in existing_runs] if existing_runs else [0]
next_run = max(run_numbers) + 1

run_dir = os.path.join(LOG_DIR, f"run_{next_run:02d}")
checkpoint_dir = os.path.join(run_dir, "checkpoints")

os.makedirs(checkpoint_dir)

print("Run directory:", run_dir)

# # Train
# predictions = Activation('softmax')(network)

# model = Model(inputs=inputs, outputs=predictions)

# model.compile(loss=CategoricalCrossentropy(),
#               optimizer=Adam(),
#               metrics=['accuracy'],
#               run_eagerly=True)

model = build_model(model_configs, network_activation, num_label, X_train.shape[1:])

def lr_schedule(epoch):
    if epoch <= 30:
        return 0.001
    elif epoch <= 100:
        return 0.0001
    else:
        return 0.00001
reduce_lr = LearningRateScheduler(lr_schedule)

# Using callback EarlyStopping
early_stopping = EarlyStopping(
    monitor='val_accuracy',  # monitor the accuracy of validation set
    patience=200,  # Allow max 5 epoch without improvement
    min_delta=0,
    mode='max',
    verbose=1,
    restore_best_weights=True,
)

checkpoint = ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, "best_model.keras"),
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1,
    validation_split=0.2,
    callbacks=[reduce_lr, early_stopping, checkpoint])

print(history.history.keys())
score = model.evaluate(X_test, y_test, verbose=0)

print("Validation Accuracy: ",max(history.history['val_accuracy']))
print("Test Loss: ", score[0])
print("Test Accuracy: ", score[1])

print("\n==========================")
model_path = os.path.join(run_dir, "tea_model.keras")
model.save(model_path)

print("Model saved:", model_path)

metrics = {
    "val_accuracy": float(max(history.history['val_accuracy'])),
    "test_accuracy": float(score[1]),
    "test_loss": float(score[0]),
    "epochs_trained": len(history.history["loss"]),
    "batch_size": BATCH_SIZE,
    "total_epochs": EPOCHS
}

with open(os.path.join(run_dir, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

Run directory: /home/nam/NPLink/SNN_framework/Software/log/6DMG/run_02


2026-03-10 16:48:55.529488: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-10 16:48:55.571514: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-10 16:48:55.571737: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Epoch 1/300
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step - accuracy: 0.1783 - loss: 2.8111
Epoch 1: val_accuracy improved from None to 0.35595, saving model to /home/nam/NPLink/SNN_framework/Software/log/6DMG/run_02/checkpoints/best_model.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 21s 323ms/step - accuracy: 0.3034 - loss: 2.4616 - val_accuracy: 0.3560 - val_loss: 1.8551 - learning_rate: 0.0010
Epoch 2/300
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step - accuracy: 0.6278 - loss: 1.5058
Epoch 2: val_accuracy improved from 0.35595 to 0.60623, saving model to /home/nam/NPLink/SNN_framework/Software/log/6DMG/run_02/checkpoints/best_model.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 17s 300ms/step - accuracy: 0.6849 - loss: 1.3010 - val_accuracy: 0.6062 - val_loss: 1.1780 - learning_rate: 0.0010
Epoch 3/300
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step - accuracy: 0.8177 - loss: 0.8258
Epoch 3: val_accuracy improved from 0.60623 to 0.75751, saving model to /home/nam/NPLink/SNN_framework/Software/log/6DMG/run_02/checkpoints/best

KeyboardInterrupt: 

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 238, 13,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 3094)      │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_3 (Lambda)   │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_4 (Lambda)   │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_5 (Lambda)   │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_6 (Lambda)   │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_7 (Lambda)   │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_8 (Lambda)   │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_9 (Lambda)   │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_10 (Lambda)  │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_11 (Lambda)  │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_12 (Lambda)  │ (None, 238)       │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_0 (Tea)       │ (None, 64)        │     30,528 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_1 (Tea)       │ (None, 64)        │     30,528 │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_2 (Tea)       │ (None, 64)        │     30,528 │ lambda_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_3 (Tea)       │ (None, 64)        │     30,528 │ lambda_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_4 (Tea)       │ (None, 64)        │     30,528 │ lambda_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_5 (Tea)       │ (None, 64)        │     30,528 │ lambda_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_6 (Tea)       │ (None, 64)        │     30,528 │ lambda_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_7 (Tea)       │ (None, 64)        │     30,528 │ lambda_7[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_8 (Tea)       │ (None, 64)        │     30,528 │ lambda_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,254,802 (4.79 MB)

 Trainable params: 314,032 (1.20 MB)

 Non-trainable params: 312,704 (1.19 MB)

 Optimizer params: 628,066 (2.40 MB)

In [ ]:
plot_model(model, to_file=TRAIN_DIR+'/model_architecture.png', show_shapes=True, show_layer_names=True)


You must install pydot (`pip install pydot`) for `plot_model` to work.
